In [7]:
# Using DistilBERT AI model

import os
import warnings
import pandas as pd
from tqdm.notebook import tqdm

# Mute standard Python warnings and Hugging Face symlink warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Load the copy of the clean, processed data directly from project folder

# 1. Define the path to the processed data file
processed_data_path = r'C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed\combined_bank_reviews_raw.csv'

# 2. Read the file into memory and immediately make an isolated copy
if os.path.exists(processed_data_path):
    df_raw = pd.read_csv(processed_data_path)
    df_clean = df_raw.copy()  # Safe copy for sentiment analysi
    print(f"✅ Successfully loaded {len(df_clean)} reviews from CSV!")
else:
    print(f"❌ Error: File not found at {processed_data_path}. Please check the path or run your scraping notebook first.")

try:
    from transformers import pipeline
    print("\n🔄 Loading DistilBERT Model (This may take a moment on the first run)...")
    sent_model = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    print("✅ Model loaded successfully!")

    # 1. Process all reviews at once for speed efficiency
    transformer_results = []
    batch_size = 16 
    
    for i in tqdm(range(0, len(df_clean), batch_size), desc="Analyzing Master Sentiment"):
        batch = df_clean['review'].iloc[i:i + batch_size].tolist()
        preds = sent_model(batch)
        transformer_results.extend(preds)

    # 2. Assign the outputs to master df_clean columns
    df_clean['transformer_sentiment_label'] = [p['label'] for p in transformer_results]
    df_clean['transformer_sentiment_score'] = [p['score'] if p['label'] == 'POSITIVE' else -p['score'] for p in transformer_results]

    # 3. ITERATE to compare distributions per bank
    print("\n" + "=" * 55)
    print(" SENTIMENT DISTRIBUTION COMPARISON BY BANK")
    print("=" * 55)
    
    # Get unique bank names dynamically present in the data
    target_banks = df_clean['bank'].unique()
    
    for bank in target_banks:
        # Isolate rows belonging only to the current bank
        df_bank = df_clean[df_clean['bank'] == bank]
        
        print(f"\n🏢 {bank.upper()}:")
        print(f"Total Cleaned Reviews: {len(df_bank)}")
        
        # Calculate raw counts and raw percentages
        counts = df_bank['transformer_sentiment_label'].value_counts()
        percentages = df_bank['transformer_sentiment_label'].value_counts(normalize=True) * 100
        
        # Print breakdown cleanly
        for label in ['POSITIVE', 'NEGATIVE']:
            cnt = counts.get(label, 0)
            pct = percentages.get(label, 0.0)
            bar = '🟩' if label == 'POSITIVE' else '🟥'
            bar_visual = bar * int(pct // 5) # Generates an inline comparative bar chart
            print(f"  {label:<10} : {cnt:>4} records ({pct:>5.1f}%)  {bar_visual}")
        
        # Calculate the average sentiment intensity score
        avg_score = df_bank['transformer_sentiment_score'].mean()
        print(f"  Average Sentiment Score (-1 to +1): {avg_score:.2f}")
        print("-" * 45)

except Exception as e:
    print(f"Transformer processing failed: {e}")

✅ Successfully loaded 1500 reviews from CSV!

🔄 Loading DistilBERT Model (This may take a moment on the first run)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded successfully!


Analyzing Master Sentiment:   0%|          | 0/94 [00:00<?, ?it/s]


 SENTIMENT DISTRIBUTION COMPARISON BY BANK

🏢 COMMERCIAL BANK OF ETHIOPIA:
Total Cleaned Reviews: 500
  POSITIVE   :  339 records ( 67.8%)  🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩
  NEGATIVE   :  161 records ( 32.2%)  🟥🟥🟥🟥🟥🟥
  Average Sentiment Score (-1 to +1): 0.37
---------------------------------------------

🏢 DASHEN BANK:
Total Cleaned Reviews: 500
  POSITIVE   :  319 records ( 63.8%)  🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩
  NEGATIVE   :  181 records ( 36.2%)  🟥🟥🟥🟥🟥🟥🟥
  Average Sentiment Score (-1 to +1): 0.28
---------------------------------------------

🏢 BANK OF ABYSSINIA:
Total Cleaned Reviews: 500
  POSITIVE   :  264 records ( 52.8%)  🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩
  NEGATIVE   :  236 records ( 47.2%)  🟥🟥🟥🟥🟥🟥🟥🟥🟥
  Average Sentiment Score (-1 to +1): 0.07
---------------------------------------------
